# Qwen3-TTS — Instant Voice Clone (Google Colab)

Clone any voice in seconds using [Qwen3-TTS](https://huggingface.co/Qwen/Qwen3-TTS-12Hz-1.7B-Base).

## How to use
1. **Runtime → Change runtime type → T4 GPU** (recommended, but CPU works too).
2. Run the single cell below (`Shift+Enter`).
3. A Gradio UI with a public share link will appear when ready.

## What it does
Upload a short reference audio clip (3–15 s) with its transcript, choose the
reference and output languages (or leave both as **Auto**), type the text you want
synthesized, and click **Generate**.  
No model-saving, no complex configuration — just instant voice cloning.


In [ ]:
# @title Qwen3-TTS — Instant Voice Clone (run this single cell)
# ── 1. Auto-install missing packages ─────────────────────────────────────────
import importlib
import subprocess
import sys

_REQUIRED = {
    "gradio":    "gradio>=4.0.0",
    "soundfile": "soundfile>=0.12.0",
    "qwen_tts":  "qwen-tts",
}

for _module, _spec in _REQUIRED.items():
    try:
        importlib.import_module(_module)
    except ImportError:
        print(f"Installing {_spec} ...")
        try:
            subprocess.run(
                [sys.executable, "-m", "pip", "install", "-q", _spec],
                check=True,
            )
        except subprocess.CalledProcessError as e:
            raise RuntimeError(
                f"Failed to install '{_spec}'.\n"
                f"Please run: pip install {_spec}\n"
                f"Original error: {e}"
            ) from e

# ── 2. GPU warning ────────────────────────────────────────────────────────────
import torch

if torch.cuda.is_available():
    print(f"✅  GPU detected: {torch.cuda.get_device_name(0)}")
else:
    print("=" * 65)
    print("⚠️   WARNING: No GPU detected — running on CPU!")
    print()
    print("   Inference on CPU is significantly slower.")
    print("   For a much faster experience, please go to:")
    print("   Runtime → Change runtime type → T4 GPU")
    print("=" * 65)

# ── 3. Engine ─────────────────────────────────────────────────────────────────
import os

import numpy as np
import soundfile as sf

_MODEL_ID = "Qwen/Qwen3-TTS-12Hz-1.7B-Base"
_MIN_DURATION = 3.0   # seconds
_MAX_DURATION = 15.0  # seconds

# All languages supported by Qwen3-TTS; "Auto" lets the model detect automatically
_SUPPORTED_LANGUAGES = [
    "Japanese", "Chinese", "English", "Korean",
    "German", "French", "Russian", "Portuguese", "Spanish", "Italian",
]
_LANGUAGE_CHOICES = ["Auto"] + _SUPPORTED_LANGUAGES


def _get_device() -> str:
    if os.environ.get("FORCE_CPU", "").lower() in ("1", "true", "yes"):
        return "cpu"
    return "cuda:0" if torch.cuda.is_available() else "cpu"


def _get_dtype(device: str) -> torch.dtype:
    return torch.bfloat16 if "cuda" in device else torch.float32


def _get_attn(device: str) -> str:
    if "cuda" in device:
        try:
            import flash_attn  # noqa: F401
            return "flash_attention_2"
        except ImportError:
            return "sdpa"
    return "sdpa"


class _TTSEngine:
    def __init__(self):
        self._model = None

    def _load(self):
        if self._model is not None:
            return
        from qwen_tts import Qwen3TTSModel
        device = _get_device()
        print(f"Loading model on {device} ...")
        self._model = Qwen3TTSModel.from_pretrained(
            _MODEL_ID,
            device_map=device,
            dtype=_get_dtype(device),
            attn_implementation=_get_attn(device),
        )
        print("Model loaded.")

    def synthesize(
        self,
        text: str,
        ref_audio_path: str,
        ref_text: str,
        ref_lang: str | None,
        output_lang: str | None,
        temperature: float = 0.65,
        repetition_penalty: float = 1.15,
        top_p: float = 0.9,
        top_k: int = 50,
    ) -> tuple:
        """Synthesize speech using a reference audio clip for voice cloning.

        Args:
            text: The text to be spoken by the cloned voice.
            ref_audio_path: Path to the reference audio file (3–15 s).
            ref_text: Exact transcript of the reference audio.
            ref_lang: Language spoken in the reference clip, or None for auto-detect.
            output_lang: Desired language for the synthesized speech, or None for auto.
            temperature: Controls expressiveness (higher = more varied, 0.3–1.3).
            repetition_penalty: Suppresses repeated syllables (1.0–1.5).
            top_p: Nucleus sampling threshold (0.8–1.0).
            top_k: Number of top candidate tokens to sample from (10–50).

        Returns:
            Tuple of (wav: np.ndarray, sample_rate: int).
        """
        self._load()
        kwargs = dict(
            text=text,
            language=output_lang,
            ref_audio=ref_audio_path,
            ref_text=ref_text,
            temperature=temperature,
            repetition_penalty=repetition_penalty,
            top_p=top_p,
            top_k=top_k,
        )
        if ref_lang is not None:
            kwargs["ref_language"] = ref_lang
        wavs, sr = self._model.generate_voice_clone(**kwargs)
        return wavs[0], sr


_engine = _TTSEngine()


def _validate_audio(path: str) -> str | None:
    """Return an error string if the audio is outside the allowed duration, else None."""
    try:
        with sf.SoundFile(path) as f:
            duration = len(f) / f.samplerate
    except Exception as exc:
        return f"Could not read audio file: {exc}"
    if duration < _MIN_DURATION or duration > _MAX_DURATION:
        return (
            f"Reference audio is {duration:.1f} s. "
            f"It must be between {_MIN_DURATION:.0f} and {_MAX_DURATION:.0f} seconds."
        )
    return None


# ── 4. Gradio UI ──────────────────────────────────────────────────────────────
import gradio as gr


def synthesize(
    ref_audio,
    ref_text,
    ref_lang,
    synth_text,
    output_lang,
    temperature,
    repetition_penalty,
    top_p,
    top_k,
):
    # ── Input validation ──────────────────────────────────────────────────────
    if ref_audio is None:
        return "❌  Please upload a reference audio clip.", None
    if not ref_text.strip():
        return "❌  Please enter the transcript of the reference audio.", None
    if not synth_text.strip():
        return "❌  Please enter the text you want synthesized.", None

    err = _validate_audio(ref_audio)
    if err:
        return f"❌  {err}", None

    # "Auto" → None so the model uses its own auto-detection
    _ref_lang = None if ref_lang == "Auto" else ref_lang
    _output_lang = None if output_lang == "Auto" else output_lang

    # ── Generate ──────────────────────────────────────────────────────────────
    try:
        wav, sr = _engine.synthesize(
            text=synth_text.strip(),
            ref_audio_path=ref_audio,
            ref_text=ref_text.strip(),
            ref_lang=_ref_lang,
            output_lang=_output_lang,
            temperature=temperature,
            repetition_penalty=repetition_penalty,
            top_p=top_p,
            top_k=int(top_k),
        )
        return "✅  Done!", (sr, wav)
    except Exception as exc:
        return f"❌  Error: {exc}", None


with gr.Blocks(title="Qwen3-TTS Instant Voice Clone", theme=gr.themes.Soft()) as demo:

    gr.Markdown(
        "# 🎤 Qwen3-TTS — Instant Voice Clone\n"
        "Upload a **reference audio clip** (3–15 s), provide its **transcript**, "
        "choose languages (or keep **Auto**), type the text to synthesize, "
        "then click **Generate**.\n\n"
        f"Model: `{_MODEL_ID}` · Device: **{_get_device()}**"
    )

    with gr.Row():
        # ── Left column: reference audio ──────────────────────────────────────
        with gr.Column():
            gr.Markdown("### Reference Audio")
            ref_audio = gr.Audio(
                label="Reference audio clip (3–15 s)",
                type="filepath",
            )
            ref_text = gr.Textbox(
                label="Transcript of the reference audio",
                placeholder="Type exactly what is said in the reference clip...",
                lines=3,
            )
            ref_lang = gr.Dropdown(
                choices=_LANGUAGE_CHOICES,
                value="Auto",
                label="Reference audio language",
                info="Language spoken in the reference clip. Auto = detect automatically.",
            )

        # ── Right column: synthesis ────────────────────────────────────────────
        with gr.Column():
            gr.Markdown("### Text to Synthesize")
            synth_text = gr.Textbox(
                label="Text to synthesize",
                placeholder="Type the text you want the cloned voice to say...",
                lines=5,
            )
            output_lang = gr.Dropdown(
                choices=_LANGUAGE_CHOICES,
                value="Auto",
                label="Output language",
                info="Language for the synthesized speech. Auto = infer from the text.",
            )
            generate_btn = gr.Button("🎵  Generate", variant="primary")

    with gr.Accordion("Advanced generation parameters", open=False):
        with gr.Row():
            temperature = gr.Slider(
                0.30, 1.30, value=0.65, step=0.05,
                label="Temperature",
                info="Higher = more expressive but less stable.",
            )
            top_p = gr.Slider(
                0.80, 1.00, value=0.90, step=0.05,
                label="Top-p",
                info="Nucleus sampling probability threshold.",
            )
        with gr.Row():
            top_k = gr.Slider(
                10, 50, value=50, step=1,
                label="Top-k",
                info="Number of top tokens to sample from.",
            )
            rep_penalty = gr.Slider(
                1.00, 1.50, value=1.15, step=0.05,
                label="Repetition Penalty",
                info="Higher = fewer repeated syllables.",
            )

    status = gr.Textbox(label="Status", interactive=False)
    audio_out = gr.Audio(label="Generated audio", type="numpy")

    generate_btn.click(
        fn=synthesize,
        inputs=[ref_audio, ref_text, ref_lang, synth_text, output_lang,
                temperature, rep_penalty, top_p, top_k],
        outputs=[status, audio_out],
    )

# ── Launch (share=True gives a public URL in Colab) ───────────────────────────
demo.launch(share=True, debug=False)
